# W06 — Validation and Research Claim Audit

**Lane:** Content-Day Performance / Decline Risk. This notebook audits the Week-5 model rather than creating a new target.

The target is a proxy: a ≥20% decrease in GSC clicks per measured day from March 1–15 to March 16–31. The model's headline test is grouped by client. The random-row split is a diagnostic showing what happens when client overlap is allowed.

**Paper review requirement:** the assignment card refers to a research paper linked in its resources. The exact paper is not included in the task text uploaded here. Do not invent its findings. Before submitting, insert the exact paper title/URL and quote or summarize two specific findings from that paper in Section 1, then answer the methodology questions.

## 1. Two paper findings and methodology questions

### Finding 1 — [insert the paper's exact finding]
- **What the paper reports:** [state the finding accurately and cite the section/table/figure]
- **Constructive methodology question:** How was the outcome/label defined, and was it measured independently of the model inputs? Could any feature contain information derived from the label?

### Finding 2 — [insert the paper's exact finding]
- **What the paper reports:** [state the finding accurately and cite the section/table/figure]
- **Constructive methodology question:** What validation design was used? Does the split prevent related clients/content or future observations from appearing on both sides? Does it support the scope of the paper's claim?

Keep the tone respectful: these are questions to understand the evidence, not accusations. Replace both placeholders before submission.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "work").exists():
    REPO_ROOT = Path("/content/FlyRank_Internship")
OUTPUT_DIR = REPO_ROOT / "work" / "outputs"
metrics_path = OUTPUT_DIR / "w05_metrics.json"
assert metrics_path.exists(), "Run work/notebooks/w05_model.ipynb first and retain its output metrics."
metrics = json.loads(metrics_path.read_text())
comparison = pd.DataFrame(metrics["comparison"])
display(comparison)
random_diag = metrics["random_split_logistic_diagnostic"]
print("Random split diagnostic:", random_diag)
print("Grouped split test clients:", metrics["test_clients"])
assert random_diag["client_overlap"] > 0, "Random split unexpectedly has no client overlap; inspect the split."

AssertionError: Run work/notebooks/w05_model.ipynb first and retain its output metrics.

## 2. Re-run the model under an honest split: before / after

The random-row split is a diagnostic only; related pages from the same client may occur in both partitions. The grouped-client holdout is the headline evaluation because it tests generalization to clients absent from training.

Compare the Logistic Regression values below. A gap is evidence that random splitting may have benefited from client overlap; it is not proof of the exact mechanism. Report both base rates because prevalence can differ across splits.

In [ ]:
grouped = comparison.loc[comparison.method == "Logistic Regression"].iloc[0].to_dict()
before_after = pd.DataFrame([
    {"design":"Random row split (diagnostic)", **{k:v for k,v in random_diag.items() if k in ["base_rate","average_precision","roc_auc","precision_at_10","precision_at_20","precision_at_50"]}},
    {"design":"Grouped client holdout (headline)", **{k:v for k,v in grouped.items() if k in ["base_rate","average_precision","roc_auc","precision_at_10","precision_at_20","precision_at_50"]}}
])
display(before_after)
before_after.to_csv(OUTPUT_DIR / "w06_split_comparison.csv", index=False)
print("Random split client overlap:", random_diag["client_overlap"])
print("Grouped split client overlap: 0 (asserted in Week 5)")

## 3. Leakage audit

The model's feature list is restricted to values knowable by the March 16 prediction cutoff. The later-window clicks define the proxy label only. Pseudonymous identifiers are grouping keys, never model features.

The audit below is a structural check. It cannot prove every upstream transformation is safe; review the SQL window boundaries and data dictionary as well.

In [ ]:
feature_names = set(metrics["features"])
forbidden = {
    "click_change_pct","decline_label","late_clicks","late_impressions",
    "late_clicks_per_day","trend_pct","trend_direction","is_declining_label",
    "health_score","priority_score","action_type","client_hash_id","content_hash_id"
}
leakage_hits = sorted(feature_names & forbidden)
assert not leakage_hits, f"Forbidden feature(s): {leakage_hits}"
assert metrics["feature_window"][1] < metrics["outcome_window"][0]
assert metrics["split"] == "GroupShuffleSplit by client_hash_id"
audit = pd.DataFrame([
    ["Feature/outcome chronology","PASS","Features end before outcome begins."],
    ["Label-derived features","PASS" if not leakage_hits else "FAIL",str(leakage_hits)],
    ["Product flags/scores","PASS","Not included in the Week-5 feature list."],
    ["Client grouping","PASS","Grouped holdout; no client overlap asserted in Week 5."],
    ["Base rate","PASS","Printed next to every Week-5 metric."],
    ["Out-of-sample evaluation","PASS","Metrics calculated on held-out clients."]
],columns=["check","status","evidence"])
display(audit)
audit.to_csv(OUTPUT_DIR / "w06_leakage_audit.csv",index=False)

## 4. Rewrite claims to match the evidence

Use only language supported by the tables:

- **Acceptable:** "In this March 2026 evaluation, the model ranked pages with the decline proxy at [measured Precision@K] on clients held out from training."
- **Not acceptable:** "The model predicts content decay for all FlyRank clients" (the validation covers one within-month split).
- **Not acceptable:** "Refreshing these pages will recover clicks" (no intervention or causal design was run).
- **Required limitation:** the label is a proxy built from a two-half comparison; it can be affected by seasonality, query mix, SERP changes, tracking availability, and small denominators.

Replace bracketed values only with the actual executed Week-5 metrics. If a metric is worse than the baseline, report that honestly.

## 5. Failure examples

Week 5 exports held-out predictions to `work/outputs/w05_test_predictions.csv`. Inspect at least three false positives and three false negatives, then annotate why each may be difficult: small denominators, partial measurement coverage, seasonality, query-mix shifts, or position/SERP changes. Do not publish raw identifiers or page-level rows in the public paper.

In [ ]:
pred_path = OUTPUT_DIR / "w05_test_predictions.csv"
if pred_path.exists():
    pred = pd.read_csv(pred_path)
    pred["predicted_decline"] = (pred.logistic_score >= 0.5).astype(int)
    fp = pred[(pred.decline_label == 0) & (pred.predicted_decline == 1)].head(3).assign(error="false positive")
    fn = pred[(pred.decline_label == 1) & (pred.predicted_decline == 0)].head(3).assign(error="false negative")
    examples = pd.concat([fp,fn],ignore_index=True)
    display(examples[["error","decline_label","logistic_score","click_change_pct"]])
    print("Add a human-written difficulty explanation for each displayed case before submission.")
else:
    print("Prediction export missing. Run Week 5 first; do not fabricate failure examples.")